<a href="https://colab.research.google.com/github/Abhirup-kar/Stellar-Class-Predictor-kaggle-competition-/blob/main/stellar_prediction_pytorch_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here We use pytorch (nural model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install opendatasets

In [15]:
import pandas as pd

In [13]:
import opendatasets as od
od.download('https://www.kaggle.com/competitions/playground-series-s6e6')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle Key: ··········


100%|██████████| 58.6M/58.6M [00:00<00:00, 161MB/s]



Extracting archive ./playground-series-s6e6/playground-series-s6e6.zip to ./playground-series-s6e6


In [16]:
sub_df = pd.read_csv('/content/playground-series-s6e6/sample_submission.csv')

In [ ]:
import joblib
stellar_data = joblib.load('/content/drive/MyDrive/stellar_dataset.joblib')

In [ ]:
train_inputs = stellar_data['train_inputs']
train_targets = stellar_data['train_targets']
val_inputs = stellar_data['val_inputs']
val_targets = stellar_data['val_targets']
test_inputs = stellar_data['test_inputs']
Lencoder = joblib.load('/content/drive/MyDrive/Lencoder.joblib')

In [ ]:
import torch

X_train_tensor = torch.tensor(train_inputs.values, dtype=torch.float32)
y_train_tensor = torch.tensor(train_targets, dtype=torch.long)

X_val_tensor = torch.tensor(val_inputs.values, dtype=torch.float32)
y_val_tensor = torch.tensor(val_targets, dtype=torch.long)

X_test_tensor = torch.tensor(test_inputs.values, dtype=torch.float32)
# Note: test_targets are not available in 'stellar_data' for accuracy calculation on the test set.

In [ ]:
import torch.nn as nn

class StellarNet(nn.Module):
    def __init__(self, input_size):
        super(StellarNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)

model = StellarNet(train_inputs.shape[1])

Loss Function and Optimizer

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

**Training Loop**

In [ ]:
epochs = 75

for epoch in range(epochs):

    outputs = model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [10/75], Loss: 0.9937
Epoch [20/75], Loss: 0.9475
Epoch [30/75], Loss: 0.8772
Epoch [40/75], Loss: 0.7778
Epoch [50/75], Loss: 0.6702
Epoch [60/75], Loss: 0.5996
Epoch [70/75], Loss: 0.5696


Prediction

In [ ]:
with torch.no_grad():
    # Make predictions on the actual test inputs
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
# The 'predicted' variable now holds the predictions for your 'test_inputs'.

The `predicted` tensor contains the class indices from your model's predictions on the `test_inputs`. To see the actual class labels, we can use the `Lencoder` to inverse transform these indices.

In [ ]:
# Decode the predicted class indices into original class labels
predicted_labels = Lencoder.inverse_transform(predicted.numpy())

print("First 10 predicted labels for test_inputs:")
print(predicted_labels[:10])

# You can also get a count of each predicted class
from collections import Counter
print("\nDistribution of predicted labels:")
print(Counter(predicted_labels))

First 10 predicted labels for test_inputs:
['GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY'
 'GALAXY' 'GALAXY']

Distribution of predicted labels:
Counter({'GALAXY': 177864, 'QSO': 69571})


Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

# To assess the model's performance, we'll calculate accuracy on the validation set.
# Since 'test_targets' are not available, we cannot compute accuracy for the 'test_inputs'.
with torch.no_grad():
    val_outputs = model(X_val_tensor)
    _, val_predicted = torch.max(val_outputs, 1)

accuracy = accuracy_score(
    y_val_tensor.numpy(),
    val_predicted.numpy()
)

print(f"Validation Accuracy: {accuracy:.4f}")


Validation Accuracy: 0.7830


In [ ]:
predicted_labels

array(['GALAXY', 'GALAXY', 'GALAXY', ..., 'GALAXY', 'GALAXY', 'GALAXY'],
      dtype=object)

In [17]:
sub_df['class'] = predicted_labels

In [18]:
sub_df.to_csv('submission_NLP.csv', index=False)

Xgboost model tuning further

In [21]:
!pip install xgboost

In [22]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=700,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    random_state=42
)

param_grid = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [4, 6, 8, 10, 12],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.2, 0.3]
}

In [23]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=30,
    cv=3,
    scoring="accuracy",
    verbose=2,
    random_state=42
)

random_search.fit(train_inputs,train_targets)
print(random_search.best_params_)
print(random_search.best_score_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [13:52:29] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=1.0, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   6.8s
[CV] END colsample_bytree=1.0, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   7.7s
[CV] END colsample_bytree=1.0, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   3.2s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=3, n_estimators=1000, subsample=0.7; total time=  19.6s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=3, n_estimators=1000, subsample=0.7; total time=  19.8s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=3, n_estimators=1000, subsample=0.7; total time=  19.6s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.01, max_depth=12, min_child_weight=1, n_estimators=1000, subsample=0

In [24]:
best_model = random_search.best_estimator_

In [25]:
accuracy_score(val_targets,best_model.predict(val_inputs))

0.9686065644756213

In [26]:
test_preds = best_model.predict(test_inputs)

In [27]:
test_preds = Lencoder.inverse_transform(test_preds)

In [ ]:
sub_df['class'] = test_preds10
sub_df.to_csv("random_forest_tune_sub.csv",index = False)

In [28]:
sub_df['class'] = test_preds

In [29]:
sub_df.to_csv("xgboost_fur_tune.csv",index = False)

Further Tuning LightGBM Model

In [30]:
!pip install lightgbm --quiet

In [34]:
from lightgbm import LGBMClassifier

lgb_gpu = LGBMClassifier(
    objective='multiclass',
    num_class=3,              # Change if your number of classes differs
    boosting_type='gbdt',

    device='gpu',

    n_estimators=500,
    learning_rate=0.05,

    num_leaves=127,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    verbose=2
)

param_dist = {
    "n_estimators": [300, 500, 700,1000],
    "learning_rate": [ 0.03, 0.05, 0.1,0.3],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [-1, 10, 20],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_samples": [10, 20, 30]
}

In [37]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    estimator=lgb_gpu,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring="accuracy",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(train_inputs, train_targets)
print(search.best_params_)
print(search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2052
[LightGBM] [Info] Number of data points in the train set: 461877, number of used features: 14
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 10 dense feature groups (5.29 MB) transferred to GPU in 0.008525 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -0.425580
[LightGBM] [Info] Start training from score -1.593068
[LightGBM] [Info] Start training from score -1.942754
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth = 16
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth = 16
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth =

In [38]:
best_model = search.best_estimator_

val_preds = best_model.predict(val_inputs)

In [39]:
accuracy_score(val_targets,val_preds)

0.9693946479605092

In [40]:
test_preds_lgbm = best_model.predict(test_inputs)

In [41]:
test_preds_lgbm = Lencoder.inverse_transform(test_preds_lgbm)

In [42]:
sub_df['class'] = test_preds_lgbm

In [43]:
sub_df.to_csv("lgbm_tune.csv",index = False)

TODO

Fined Tuning Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

def tune_random_forest(
    X_train,
    y_train,
    cv=3,
    n_iter=15,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
):
    """
    Tune RandomForest using RandomizedSearchCV.

    Returns:
        best_model
        best_params
        best_score
    """

    rf = RandomForestClassifier(
        random_state=random_state,
        n_jobs=n_jobs
    )

    param_dist = {
        'n_estimators': [200,500,800],
        'max_depth': [10, 15],
        'min_samples_split': [2, 8,15],
        'min_samples_leaf': [3,8],
        'max_features': ['sqrt', 'log2'],
        'bootstrap': [True, False]
    }

    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        random_state=random_state,
        n_jobs=n_jobs,
        verbose=2
    )

    search.fit(X_train, y_train)

    return (
        search.best_estimator_,
        search.best_params_,
        search.best_score_
    )

In [ ]:
best_rf, best_params, best_score = tune_random_forest(
    train_inputs,
    train_targets,
    cv=3,
    n_iter=5
)

print("Best Parameters:")
print(best_params)

print("\nBest CV Score:")
print(best_score)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


KeyboardInterrupt: 

In [ ]:
model10_random_forest_tune = best_rf

In [ ]:
val_preds10 = model10_random_forest_tune.predict(val_inputs)
accuracy_score(val_targets,val_preds10)

In [ ]:
test_preds10 = model10_random_forest_tune.predict(test_inputs)

In [ ]:
test_preds10 = Lencoder.inverse_transform(test_preds10)